# TP2 - Sentiment Analysis (Version structuree)

Objectif: classifier des critiques de films IMDb en `positive` ou `negative`, comparer plusieurs strategies de prompting et evaluer la performance avec le score F1.

## 1) Imports et configuration

In [4]:
import json
import numpy as np
import pandas as pd
from datasets import load_dataset
from dotenv.ipython import load_dotenv
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

load_dotenv(override=True)
RANDOM_STATE = 123

## 2) Preparation des donnees

In [ ]:
ds = load_dataset("imdb")
train_df = ds["train"].to_pandas().copy()
train_df["sentiment"] = np.where(train_df["label"] == 1, "positive", "negative")

examples_df, gold_examples_df = train_test_split(
    train_df[["text", "sentiment"]],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=train_df["sentiment"],
)

print("Train examples:", examples_df.shape)
print("Gold examples:", gold_examples_df.shape)
examples_df.head(3)

Train examples: (20000, 2)
Gold examples: (5000, 2)


,text,sentiment
13586,"This movie was made in 1948, but it still ring...",positive
14461,At this point it seems almost unnecessary to s...,positive
20844,Thats My Bush is first of all a very entertain...,positive


In [6]:
def create_balanced_examples_json(dataset: pd.DataFrame, n_per_class: int = 2, seed: int = 34) -> str:
    positive = dataset[dataset["sentiment"] == "positive"].sample(n_per_class, random_state=seed)
    negative = dataset[dataset["sentiment"] == "negative"].sample(n_per_class, random_state=seed)
    sampled = pd.concat([positive, negative]).sample(frac=1.0, random_state=seed)
    return sampled[["text", "sentiment"]].to_json(orient="records")

gold_examples = gold_examples_df.sample(50, random_state=RANDOM_STATE).to_json(orient="records")
few_shot_examples = create_balanced_examples_json(examples_df, n_per_class=2, seed=34)
json.loads(few_shot_examples)[:2]

[{'text': '-SPOILES- Lame south of the border adventure movie that has something to do with the blackmail of a big cooperate executive Rosenlski the president of Unasco Inc. by on the lamb beachcomber David Ziegler who\'s living the life of Reilly, or Ziegler, in his beach house in Cancun Mexico.Having this CD, that he gave to his brother James, that has three years of phone conversations between Rosenlski and the President of the United States involved in criminal deals. This CD has given David an edge over the international mobsters who are after him. <br /><br />The fact that James get\'s a little greedy by trying to shake down Rosenlski for 2 million in diamonds not only cost him his life but put David in danger of losing his as well. Ropsenlski want\'s to negotiate with David for the CD by getting his ex-wife Liz to talk to him about giving it up, Rosnelski made a deal to pay off her debts if she comes through. David is later killed by Rosenliski\'s Mexican hit-man Tony, with the 

## 3) Construction des prompts

In [7]:
USER_PROMPT_TEMPLATE = """```{movie_review}```"""

BASE_SYSTEM_MESSAGE = """
Classify the sentiment of movie reviews presented in the input as 'positive' or 'negative'.
Movie reviews will be delimited by triple backticks in the input.
Answer only 'positive' or 'negative'.
Do not explain your answer.
"""

COT_SYSTEM_MESSAGE = """
Classify the sentiment of movie reviews presented in the input as 'positive' or 'negative'.
Movie reviews will be delimited by triple backticks in the input.
Think carefully before deciding, but output strictly one label: 'positive' or 'negative'.
Do not explain your answer.
"""

def create_prompt_messages(system_message: str, examples_json: str | None = None) -> list[dict]:
    messages = [{"role": "system", "content": system_message.strip()}]
    if examples_json is None:
        return messages

    for item in json.loads(examples_json):
        review = item["text"]
        sentiment = item["sentiment"]
        messages.append({"role": "user", "content": USER_PROMPT_TEMPLATE.format(movie_review=review)})
        messages.append({"role": "assistant", "content": sentiment})
    return messages

zero_shot_prompt = create_prompt_messages(BASE_SYSTEM_MESSAGE)
few_shot_prompt = create_prompt_messages(BASE_SYSTEM_MESSAGE, few_shot_examples)
cot_few_shot_prompt = create_prompt_messages(COT_SYSTEM_MESSAGE, few_shot_examples)

len(zero_shot_prompt), len(few_shot_prompt), len(cot_few_shot_prompt)

(1, 9, 9)

## 4) Evaluation

In [8]:
def normalize_prediction(text: str) -> str:
    value = text.strip().lower()
    if "negative" in value:
        return "negative"
    if "positive" in value:
        return "positive"
    return "unknown"

def evaluate_prompt(prompt: list[dict], gold_examples_json: str, llm) -> dict:
    y_true, y_pred = [], []

    for item in json.loads(gold_examples_json):
        review = item["text"]
        truth = item["sentiment"]
        user_input = [{"role": "user", "content": USER_PROMPT_TEMPLATE.format(movie_review=review)}]

        try:
            response = llm.invoke(prompt + user_input)
            pred = normalize_prediction(response.content)
        except Exception:
            pred = "unknown"

        y_true.append(truth)
        y_pred.append(pred)

    valid_mask = [p in {"positive", "negative"} for p in y_pred]
    filtered_true = [t for t, keep in zip(y_true, valid_mask) if keep]
    filtered_pred = [p for p, keep in zip(y_pred, valid_mask) if keep]

    f1 = f1_score(filtered_true, filtered_pred, average="micro") if filtered_true else 0.0
    coverage = float(np.mean(valid_mask))

    return {"f1_micro": f1, "coverage": coverage, "n_eval": len(y_true)}

In [19]:
# Dictionnaire final: uniquement les modeles effectivement instancies
available_models = {}
unavailable_models = {}

try:
    from langchain_openai import ChatOpenAI
    available_models["gpt-4o"] = ChatOpenAI(model="gpt-4o", temperature=0)
except Exception as e:
    unavailable_models["gpt-4o"] = str(e)

try:
    from langchain_ollama import ChatOllama
    available_models["llama3.2"] = ChatOllama(model="llama3.2", temperature=0)
except Exception as e:
    unavailable_models["llama3.2"] = str(e)

print("Modeles disponibles:", list(available_models.keys()))
if unavailable_models:
    print("Modeles indisponibles:")
    for name, err in unavailable_models.items():
        print(f"- {name}: {err}")

Modeles disponibles: ['gpt-4o', 'llama3.2']


In [20]:
prompt_variants = {
    "zero_shot": zero_shot_prompt,
    "few_shot": few_shot_prompt,
    "cot_few_shot": cot_few_shot_prompt,
}

rows = []
for model_name, model in available_models.items():
    for prompt_name, prompt in prompt_variants.items():
        metrics = evaluate_prompt(prompt, gold_examples, model)
        rows.append({
            "model": model_name,
            "prompt": prompt_name,
            **metrics,
        })

if rows:
    results_df = pd.DataFrame(rows).sort_values(["model", "f1_micro"], ascending=[True, False]).reset_index(drop=True)
else:
    results_df = pd.DataFrame(columns=["model", "prompt", "f1_micro", "coverage", "n_eval"])
    print("Aucun resultat: verifie la cellule de chargement des modeles (available_models vide).")

results_df

,model,prompt,f1_micro,coverage,n_eval
0,gpt-4o,zero_shot,0.920000,1.00,50
1,gpt-4o,few_shot,0.918367,0.98,50
2,gpt-4o,cot_few_shot,0.902439,0.82,50
3,llama3.2,zero_shot,0.840000,1.00,50
4,llama3.2,few_shot,0.840000,1.00,50
5,llama3.2,cot_few_shot,0.840000,1.00,50


## 5) Robustesse (few-shot sur plusieurs tirages)

In [21]:
if "gpt-4o" in available_models:
    llm = available_models["gpt-4o"]
    runs = 10
    scores = []

    for i in range(runs):
        sampled_examples = create_balanced_examples_json(examples_df, n_per_class=4, seed=RANDOM_STATE + i)
        sampled_prompt = create_prompt_messages(BASE_SYSTEM_MESSAGE, sampled_examples)
        metrics = evaluate_prompt(sampled_prompt, gold_examples, llm)
        scores.append(metrics["f1_micro"])

    print(f"Few-shot (GPT-4o) over {runs} runs -> mean={np.mean(scores):.4f}, std={np.std(scores):.4f}")
else:
    print("Cell skipped: gpt-4o non disponible dans cet environnement.")

Few-shot (GPT-4o) over 10 runs -> mean=0.9318, std=0.0127
